In [93]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()

In [ ]:
# import importlib
# import src.utils.cleaner

# importlib.reload(src.utils.cleaner)

<module 'src.utils.cleaner' from 'c:\\Users\\hkand\\streaming\\sales_streaming_analytics\\src\\utils\\cleaner.py'>

In [ ]:
# import sys
# import os

# home = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
# sys.path.append(home)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, LongType
from utils.json_parser import parse_json
from utils.time_utils import date_day_weekOfMonth
from utils.cleaner import good_exp_records, bad_exp_records
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = dbutils.widgets.get("catalog")
checkpoints = dbutils.widgets.get("checkpoints_dir")

df = spark.readStream.table(f"{catalog}.brz.expenses_raw")

df = df.select("value")

schema = StructType([
    StructField("expense_id", IntegerType(), True),
    StructField("employee_id", LongType(), True),
    StructField("region_id", IntegerType(), True),
    StructField("expense_type", StringType(), True),
    StructField("expense_amount", LongType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("ingestion_time", TimestampType(), True)
])

parsed_df = parse_json(spark, df, "value", schema)

parsed_df = (parsed_df.select("parsed.expense_id", "parsed.employee_id",
                                "parsed.region_id", "parsed.expense_type",
                                "parsed.expense_amount", "parsed.event_time"))

def process_expenses(batch_df, batch_id):

    w = Window.partitionBy("expense_id").orderBy(F.col("event_time").desc())

    batch_df = (batch_df.withColumn("rn", F.row_number().over(w))
                        .filter(F.col("rn")==1)
                        .drop("rn"))

    good_exp_df = good_exp_records(spark, batch_df)

    bad_exp_df = bad_exp_records(spark, batch_df)

    good_exp_df = date_day_weekOfMonth(spark, good_exp_df, "event_time")

    good_exp_df = (good_exp_df.select("expense_id", "employee_id", "region_id",
                                        "expense_type", "expense_amount", "event_time", "event_date",
                                        "day", "week_of_month", "processed_time"))

    good_exp_df.write.format("delta").mode("append").partitionBy("event_date").saveAsTable(f"{catalog}.slv.expenses")

    bad_exp_df.write.format("delta").mode("append").saveAsTable(f"{catalog}.brz.bad_expenses_records")

query = (parsed_df.writeStream
                    .foreachBatch(process_expenses)
                    .option("checkpointLocation", f"abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/{checkpoints}/slv_checkpoints/expenses_checkpoint")
                    .trigger(availableNow = True)
                    .start())

query.awaitTermination()